## Bloque 1 — Carga y primer vistazo

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/training-data.csv")
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [2]:
df.shape

(150000, 12)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

In [4]:
df.describe()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [5]:
df.columns

Index(['Unnamed: 0', 'SeriousDlqin2yrs',
       'RevolvingUtilizationOfUnsecuredLines', 'age',
       'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
       'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       'NumberOfDependents'],
      dtype='str')

## Bloque 2 — Variable objetivo (`SeriousDlqin2yrs`)

In [6]:
df['SeriousDlqin2yrs'].value_counts()

SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

In [7]:
df['SeriousDlqin2yrs'].value_counts(normalize=True)

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

> **Interpretación:** confirma el desbalance de clases (~6-7% de mora / default).
> Esto condiciona las métricas a usar más adelante (no usar accuracy como métrica principal) y obliga a estratificar los splits train/test.

## Bloque 3 — Calidad de datos

### 3.1 `MonthlyIncome` — missing

In [8]:
df['MonthlyIncome'].unique()[:20]

array([ 9120.,  2600.,  3042.,  3300., 63588.,  3500.,    nan, 23684.,
        2500.,  6501., 12454., 13700.,     0., 11362.,  8800.,  3280.,
         333., 12300.,  3000.,  7916.])

In [9]:
df['MonthlyIncome'].dtype

dtype('float64')

In [10]:
df['MonthlyIncome'].isnull().sum()

np.int64(29731)

In [11]:
df['MonthlyIncome'].isnull().mean() * 100

np.float64(19.820666666666668)

In [12]:
# Tasa de mora comparando clientes con y sin missing en MonthlyIncome
tasa_mora_con_missing = df[df['MonthlyIncome'].isnull()]['SeriousDlqin2yrs'].mean()
tasa_mora_sin_missing = df[~df['MonthlyIncome'].isnull()]['SeriousDlqin2yrs'].mean()

print("Tasa de mora (con missing en ingreso):", tasa_mora_con_missing)
print("Tasa de mora (sin missing en ingreso):", tasa_mora_sin_missing)

Tasa de mora (con missing en ingreso): 0.05613669234132723
Tasa de mora (sin missing en ingreso): 0.06948590243537403


In [13]:
# Edad promedio comparando ambos grupos
edad_con_missing = df[df['MonthlyIncome'].isnull()]['age'].mean()
edad_sin_missing = df[~df['MonthlyIncome'].isnull()]['age'].mean()

print("Edad promedio (con missing en ingreso):", edad_con_missing)
print("Edad promedio (sin missing en ingreso):", edad_sin_missing)

Edad promedio (con missing en ingreso): 56.362349063267295
Edad promedio (sin missing en ingreso): 51.289792049489066


> **Conclusión:** `MonthlyIncome` tiene 19.8% de missing. El grupo con missing muestra una tasa de mora menor (5.6% vs 6.9%) y una edad promedio mayor (56 vs 51 años), lo que sugiere que gran parte de este missing corresponde a clientes jubilados con ingresos estables no capturados como "salario mensual", no a informalidad de alto riesgo. **Decisión:** se tratará como categoría propia en el binning WOE, no se imputará.

### 3.2 `NumberOfDependents` — missing

In [14]:
NumberOfDependents_with_missing = df[df['NumberOfDependents'].isnull()]['SeriousDlqin2yrs'].mean()
NumberOfDependents_with_no_missing = df[~df['NumberOfDependents'].isnull()]['SeriousDlqin2yrs'].mean()

print(NumberOfDependents_with_missing)
print(NumberOfDependents_with_no_missing)

0.04561671763506626
0.0674101152824557


In [15]:
edad_con_missing_dep = df[df['NumberOfDependents'].isnull()]['age'].mean()
edad_sin_missing_dep = df[~df['NumberOfDependents'].isnull()]['age'].mean()

print("Edad promedio (con missing en dependientes):", edad_con_missing_dep)
print("Edad promedio (sin missing en dependientes):", edad_sin_missing_dep)

Edad promedio (con missing en dependientes): 59.58893985728848
Edad promedio (sin missing en dependientes): 52.09927708863879


> **Conclusión:** `NumberOfDependents` tiene 2.6% de missing. El grupo con missing muestra una tasa de mora menor (4.56% vs 6.74%) y una edad promedio mayor (59.6 vs 52.1 años), consistente con el patrón observado en `MonthlyIncome`. **Decisión:** categoría propia en el binning WOE, no se imputará.

In [16]:
# Tasa de mora según número de dependientes (solo para explorar el patrón, no missing)
df.groupby('NumberOfDependents')['SeriousDlqin2yrs'].agg(['mean', 'count'])

,mean,count
NumberOfDependents,,
0.0,0.058629,86902
1.0,0.073529,26316
2.0,0.081139,19522
3.0,0.088263,9483
4.0,0.103774,2862
5.0,0.091153,746
6.0,0.151899,158
7.0,0.098039,51
8.0,0.083333,24


> **Conclusión:** la tasa de mora aumenta de forma consistente con el número de dependientes reportados (5.9% con 0 dependientes hasta 15.2% con 6), coherente con mayor presión financiera por persona a cargo. Los valores de 7+ dependientes tienen muy poco volumen (≤51 casos, algunos con solo 1-5 observaciones) por lo que sus tasas de mora no son estadísticamente confiables y se agruparán en un bin conjunto en la Fase 2.

### 3.3 `age = 0`

In [17]:
df['age'].describe()

count    150000.000000
mean         52.295207
std          14.771866
min           0.000000
25%          41.000000
50%          52.000000
75%          63.000000
max         109.000000
Name: age, dtype: float64

In [18]:
print("El numero de casos con age = 0 es: ", (df['age'] == 0).sum())

El numero de casos con age = 0 es:  1


In [19]:
df[df['age'] == 0]

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65695,65696,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


> **Conclusión:** 1 caso con `age = 0` (fila índice 65695) de 150,000 observaciones. El resto de variables de esa fila están dentro de rangos normales, lo que sugiere un error de captura aislado en el campo edad, no un problema sistemático. **Decisión:** se elimina la fila del dataset.

In [20]:
df = df[df['age'] != 0]

### 3.4 `RevolvingUtilizationOfUnsecuredLines` — valores extremos (EN CURSO)

In [21]:
df['RevolvingUtilizationOfUnsecuredLines'].describe()

count    149999.000000
mean          6.048472
std         249.756203
min           0.000000
25%           0.029867
50%           0.154176
75%           0.559044
max       50708.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64

In [22]:
(df['RevolvingUtilizationOfUnsecuredLines'] > 1).sum()

np.int64(3321)

In [23]:
(df['RevolvingUtilizationOfUnsecuredLines'] > 2).sum()

np.int64(371)

In [24]:
(df['RevolvingUtilizationOfUnsecuredLines'] > 10).sum()

np.int64(241)

In [25]:
df[df['RevolvingUtilizationOfUnsecuredLines'] > 10].head(10)

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
293,294,0,2340.0,45,0,0.339333,8333.0,7,0,2,0,2.0
697,698,1,2066.0,58,0,0.271121,6000.0,8,1,1,0,1.0
1991,1992,1,1143.0,44,2,0.547762,6500.0,13,0,4,0,2.0
2331,2332,0,6324.0,59,0,0.136673,11318.0,4,0,1,0,1.0
4278,4279,0,1982.0,33,0,0.144982,8000.0,4,0,0,0,0.0
4793,4794,0,3746.0,52,2,0.481353,2600.0,5,0,1,0,3.0
6760,6761,0,941.0,37,0,0.085183,5000.0,6,0,0,0,1.0
6850,6851,0,8497.0,28,0,0.404930,5800.0,5,0,1,0,0.0
7637,7638,1,1577.0,37,0,0.479826,5526.0,11,0,2,0,2.0
7774,7775,0,1028.0,27,0,442.000000,NaN,2,0,0,0,0.0


> **Pendiente:** confirmar hipótesis sobre por qué esta variable tiene valores extremos en filas con perfil normal (¿error de escala/unidad?), investigar la relación entre `DebtRatio` extremo (ej. fila índice 7774, `DebtRatio = 442`) y `MonthlyIncome` nulo, y decidir tratamiento final (capar / flag de anómalo / ambos).
>
> **Siguientes pasos (no completados aún en esta sesión):**
> - Análisis dedicado de `DebtRatio` con `.describe()`.
> - Detectar códigos 96/98 en las variables de atrasos históricos (`NumberOfTime30-59...`, `60-89...`, `NumberOfTimes90DaysLate`).
> - Tabla resumen de calidad de datos.
> - Bloque 4: análisis bivariado (tasa de mora por bins de cada variable clave).

Análisis dedicado de `DebtRatio`

In [26]:
df['DebtRatio'].describe()

count    149999.000000
mean        353.007426
std        2037.825113
min           0.000000
25%           0.175074
50%           0.366503
75%           0.868257
max      329664.000000
Name: DebtRatio, dtype: float64

In [28]:
df[df['MonthlyIncome'].isnull()]['DebtRatio'].describe()


count     29731.000000
mean       1673.396556
std        4248.372895
min           0.000000
25%         123.000000
50%        1159.000000
75%        2382.000000
max      329664.000000
Name: DebtRatio, dtype: float64

In [29]:
df[~df['MonthlyIncome'].isnull()]['DebtRatio'].describe()

count    120268.000000
mean         26.598995
std         424.448215
min           0.000000
25%           0.143388
50%           0.296021
75%           0.482560
max       61106.500000
Name: DebtRatio, dtype: float64

> **Conclusión `DebtRatio`:** los valores extremos de `DebtRatio` están fuertemente concentrados en las filas donde `MonthlyIncome` es nulo (percentil 75 = 2,382 en ese grupo, vs 0.48 en el grupo con ingreso reportado). Esto confirma que gran parte de estos valores absurdos son consecuencia matemática de dividir por un ingreso no capturado (o casi cero), no de clientes reales con ese nivel de apalancamiento.
>
> **Decisión:** se capará `DebtRatio` a un máximo razonable (ej. 2-3, tal como sugiere el documento del proyecto) y se creará un flag `DebtRatio_anomalo` para las filas afectadas, en vez de tratarlas como apalancamiento real. Este tratamiento estará naturalmente ligado a la categoría "missing" que ya se decidió para `MonthlyIncome`.

Detectar códigos 96/98 en las variables de atrasos históricos

In [30]:
df['NumberOfTime30-59DaysPastDueNotWorse'].value_counts()

NumberOfTime30-59DaysPastDueNotWorse
0     126018
1      16032
2       4598
3       1754
4        747
5        342
98       264
6        140
7         54
8         25
9         12
96         5
10         4
12         2
13         1
11         1
Name: count, dtype: int64

In [31]:
df['NumberOfTime60-89DaysPastDueNotWorse'].value_counts()

NumberOfTime60-89DaysPastDueNotWorse
0     142395
1       5731
2       1118
3        318
98       264
4        105
5         34
6         16
7          9
96         5
8          2
11         1
9          1
Name: count, dtype: int64

In [32]:
df['NumberOfTimes90DaysLate'].value_counts()

NumberOfTimes90DaysLate
0     141661
1       5243
2       1555
3        667
4        291
98       264
5        131
6         80
7         38
8         21
9         19
10         8
96         5
11         5
13         4
15         2
14         2
12         2
17         1
Name: count, dtype: int64

In [34]:
df[df['NumberOfTime30-59DaysPastDueNotWorse'].isin([96, 98])].shape[0]


269

In [35]:
df[df['NumberOfTime60-89DaysPastDueNotWorse'].isin([96, 98])].shape[0]

269

In [ ]:
df[df['NumberOfTimes90DaysLate'].isin([96, 98])].shape[0]

269

In [37]:
df[(df['NumberOfTime30-59DaysPastDueNotWorse'].isin([96, 98])) & 
   (df['NumberOfTime60-89DaysPastDueNotWorse'].isin([96, 98])) & 
   (df['NumberOfTimes90DaysLate'].isin([96, 98]))].shape[0]

269

In [38]:
tasa_mora_codigos_raros = df[df['NumberOfTimes90DaysLate'].isin([96, 98])]['SeriousDlqin2yrs'].mean()
tasa_mora_normal = df[~df['NumberOfTimes90DaysLate'].isin([96, 98])]['SeriousDlqin2yrs'].mean()

print("Tasa de mora (codigos raros):", tasa_mora_codigos_raros)
print("Tasa de mora (normal):", tasa_mora_normal)

Tasa de mora (codigos raros): 0.5464684014869888
Tasa de mora (normal): 0.06597876177118814


> **Conclusión — códigos 96/98 en variables de atraso histórico:** las tres variables (`NumberOfTime30-59...`, `60-89...`, `NumberOfTimes90DaysLate`) comparten los mismos códigos anómalos (96 y 98), con 264 y 5 casos respectivamente. Estos códigos no representan conteos reales de atraso, sino un código de sistema (probablemente asociado a cuentas con historial muy problemático o sin cálculo estándar posible). La tasa de mora de este grupo es de 54%, muy por encima del 6.6% del resto de la población — más de 8 veces superior.
>
> **Decisión:** no se eliminan ni se imputan. Se tratarán como categoría/bin propio en el binning WOE de la Fase 2, dado su fuerte poder discriminante (alto riesgo), en vez de mezclarlos con la escala numérica normal de atrasos.

In [40]:
resumen_calidad = pd.DataFrame({
    'variable': [
        'MonthlyIncome',
        'NumberOfDependents',
        'age',
        'RevolvingUtilizationOfUnsecuredLines',
        'DebtRatio',
        'NumberOfTime30-59DaysPastDueNotWorse / 60-89 / 90DaysLate'
    ],
    'problema': [
        'Missing (19.8%)',
        'Missing (2.6%)',
        '1 valor imposible (age = 0)',
        'Valores extremos (>10), imposibles como ratio real',
        'Valores extremos, asociados a MonthlyIncome nulo',
        'Códigos de sistema 96/98 en vez de conteos reales'
    ],
    'n_filas_afectadas': [
        29731,
        3924,
        1,
        241,
        'Concentrado en filas con MonthlyIncome nulo',
        264 + 5  # o el número exacto de filas únicas que confirmaste
    ],
    'decision': [
        'No imputar. Categoría propia ("missing") en binning WOE',
        'No imputar. Categoría propia ("missing") en binning WOE',
        'Eliminar la fila',
        'Capar a un máximo razonable + flag de "valor anómalo"',
        'Capar a un máximo razonable + flag de "valor anómalo"',
        'No eliminar. Categoría/bin propio en binning WOE'
    ],
    'justificacion': [
        'Grupo con missing tiene menor mora (5.6% vs 6.9%) y mayor edad (56 vs 51). Asociado a jubilados con ingreso no capturado como salario, no a informalidad de riesgo.',
        'Mismo patrón que MonthlyIncome: menor mora (4.6% vs 6.7%) y mayor edad (59.6 vs 52.1). Hijos ya independientes.',
        'Único caso, resto de variables de la fila normales. Error de captura aislado, no sistemático.',
        'Media (6.05) muy por encima de mediana (0.15). Valores >10 (0.16% del dataset) económicamente imposibles como ratio de utilización.',
        'Percentil 75 es 2,382 en filas con ingreso nulo vs 0.48 en filas con ingreso reportado. Confirma que el missing en ingreso "rompe" el cálculo del ratio.',
        'Tasa de mora del grupo es 54% vs 6.6% del resto (~8x superior). El código en sí es señal de alto riesgo, no ruido — eliminarlo perdería la información más predictiva del dataset.'
    ]
})
resumen_calidad

,variable,problema,n_filas_afectadas,decision,justificacion
0,MonthlyIncome,Missing (19.8%),29731,"No imputar. Categoría propia (""missing"") en bi...",Grupo con missing tiene menor mora (5.6% vs 6....
1,NumberOfDependents,Missing (2.6%),3924,"No imputar. Categoría propia (""missing"") en bi...",Mismo patrón que MonthlyIncome: menor mora (4....
2,age,1 valor imposible (age = 0),1,Eliminar la fila,"Único caso, resto de variables de la fila norm..."
3,RevolvingUtilizationOfUnsecuredLines,"Valores extremos (>10), imposibles como ratio ...",241,"Capar a un máximo razonable + flag de ""valor a...",Media (6.05) muy por encima de mediana (0.15)....
4,DebtRatio,"Valores extremos, asociados a MonthlyIncome nulo",Concentrado en filas con MonthlyIncome nulo,"Capar a un máximo razonable + flag de ""valor a...","Percentil 75 es 2,382 en filas con ingreso nul..."
5,NumberOfTime30-59DaysPastDueNotWorse / 60-89 /...,Códigos de sistema 96/98 en vez de conteos reales,269,No eliminar. Categoría/bin propio en binning WOE,Tasa de mora del grupo es 54% vs 6.6% del rest...


## Bloque 2 — Análisis Bivariado

Variable AGE

In [ ]:
df['age_bin'] = pd.qcut(df['age'], q=10)
df.groupby('age_bin')['SeriousDlqin2yrs'].agg(['mean', 'count'])

> **Conclusión — `age`:** la tasa de mora decrece de forma monotónica con la edad, desde 11.4% en el decil más joven (21-33 años) hasta 2.2% en el decil de mayor edad (72-109 años), sin ningún tramo de excepción. A diferencia de lo planteado inicialmente, no se observa estabilización ni repunte en edades avanzadas — la relación es limpiamente decreciente en todo el rango. Esto confirma a `age` como una variable con fuerte poder predictivo y relación monotónica, ideal para el binning WOE de la Fase 2 (sin necesidad de bins adicionales por comportamiento no lineal).

Variable DEBTRATIO

In [ ]:
df['debtratio_bin'] = pd.qcut(df['DebtRatio'], q=10)
df.groupby('debtratio_bin')['SeriousDlqin2yrs'].agg(['mean', 'count'])

In [44]:
df['DebtRatio_anomalo'] = (df['DebtRatio'] > 2).astype(int)

In [45]:
df['DebtRatio_capped'] = df['DebtRatio'].clip(upper=2)

In [ ]:
df['debtratio_bin'] = pd.qcut(df['DebtRatio_capped'], q=10, duplicates='drop')
df.groupby('debtratio_bin')['SeriousDlqin2yrs'].agg(['mean', 'count'])

In [47]:
(df['DebtRatio_capped'] == 2).sum()

np.int64(31215)

> **Conclusión — `DebtRatio`:** la variable original tiene una cola extrema (percentil 75 general vs máximo de 329,664) fuertemente concentrada en filas con `MonthlyIncome` nulo, aunque no exclusivamente (31,215 filas superan el cap de 2, más que las 29,731 filas con ingreso nulo). Se aplicó un cap en 2.0 (`DebtRatio_capped`) y un flag binario (`DebtRatio_anomalo`) para preservar la señal de "valor anómalo" sin perderla en el capping. El binning exploratorio con `qcut` confirma que, tras el cap, el 69% del bin superior son valores idénticos (el propio tope de 2.0), por lo que en la Fase 2 el binning definitivo (`optbinning`) deberá tratar este punto de masa de forma explícita, apoyándose en el flag `DebtRatio_anomalo` para separar señal real de artefacto matemático.

Variable RevolvingUtilizationOfUnsecuredLines

In [54]:
df['RevolvingUtilizationOfUnsecuredLines_anomalo'] = (df['RevolvingUtilizationOfUnsecuredLines'] > 1).astype(int)

In [55]:
df['RevolvingUtilizationOfUnsecuredLines_capped'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1)

In [56]:
df['revol_bin'] = pd.qcut(df['RevolvingUtilizationOfUnsecuredLines_capped'], q=10, duplicates='drop')
df.groupby('revol_bin')['SeriousDlqin2yrs'].agg(['mean', 'count'])

,mean,count
revol_bin,,
"(-0.001, 0.00297]",0.025200,15000
"(0.00297, 0.0192]",0.013600,15000
"(0.0192, 0.0435]",0.014000,15000
"(0.0435, 0.0832]",0.019133,15000
"(0.0832, 0.154]",0.023733,15000
"(0.154, 0.271]",0.034736,14999
"(0.271, 0.445]",0.052467,15000
"(0.445, 0.699]",0.088000,15000
"(0.699, 0.981]",0.165800,15000


In [57]:
(df['RevolvingUtilizationOfUnsecuredLines_capped'] == 1).sum()

np.int64(3338)

In [58]:
edad_utilizacion_baja = df[df['RevolvingUtilizationOfUnsecuredLines_capped'] <= 0.00297]['age'].mean()
edad_resto = df[df['RevolvingUtilizationOfUnsecuredLines_capped'] > 0.00297]['age'].mean()

print("Edad promedio (utilización ~0):", edad_utilizacion_baja)
print("Edad promedio (resto):", edad_resto)

Edad promedio (utilización ~0): 54.5913605759616
Edad promedio (resto): 52.04044504363028


In [59]:
ingreso_utilizacion_baja = df[df['RevolvingUtilizationOfUnsecuredLines_capped'] <= 0.00297]['MonthlyIncome'].mean()
ingreso_resto = df[df['RevolvingUtilizationOfUnsecuredLines_capped'] > 0.00297]['MonthlyIncome'].mean()

print("Ingreso promedio (utilización ~0):", ingreso_utilizacion_baja)
print("Ingreso promedio (resto):", ingreso_resto)

Ingreso promedio (utilización ~0): 6510.7257005456395
Ingreso promedio (resto): 6685.983838106985


> **Conclusión — `RevolvingUtilizationOfUnsecuredLines`:** tras aplicar un cap en 1.0 (100% de utilización) y un flag `_anomalo` para valores originales >1, el 22% del bin superior corresponde al punto de masa del propio cap (3,338 de 15,000 filas). El patrón de mora es mayormente creciente con la utilización (de 1.4% a 23.2%), como se esperaría económicamente — a mayor uso relativo del crédito disponible, mayor riesgo. Sin embargo, el bin de utilización casi nula (0-0.3%) muestra una mora ligeramente mayor que el siguiente bin (2.5% vs 1.4%), rompiendo la monotonicidad estricta. Este grupo tiene edad promedio mayor e ingreso reportado menor que el resto, consistente con el patrón ya observado en `MonthlyIncome` — sugiere una submuestra de clientes jubilados/con ingresos no convencionales, cuyo comportamiento de bajo uso de crédito no refleja necesariamente bajo riesgo.
>
> **Implicación para Fase 2:** esta variable tendrá una relación mayormente monotónica pero con una posible excepción en el extremo inferior — el binning supervisado de `optbinning` deberá decidir si fuerza monotonicidad total (fusionando ese primer bin con el siguiente) o si se permite esta única excepción justificada por evidencia de negocio.

Variable: NumberOfOpenCreditLinesAndLoans

In [61]:
df.groupby('NumberOfOpenCreditLinesAndLoans')['SeriousDlqin2yrs'].agg(['mean', 'count'])

,mean,count
NumberOfOpenCreditLinesAndLoans,,
0,0.256356,1888
1,0.131816,4438
2,0.091359,6666
3,0.075293,9058
4,0.064002,11609
5,0.063336,12931
6,0.055388,13613
7,0.056399,13245
8,0.048241,12562


In [62]:
df['NumberOfOpenCreditLinesAndLoans_anomalo'] = (df['NumberOfOpenCreditLinesAndLoans'] >= 29).astype(int)
df['NumberOfOpenCreditLinesAndLoans_capped'] = df['NumberOfOpenCreditLinesAndLoans'].clip(upper=29)

> **Conclusión — `NumberOfOpenCreditLinesAndLoans`:** se confirma una relación en forma de U parcial. La mora es más alta en el extremo de pocas líneas de crédito (25.6% con 0 líneas, cayendo a un mínimo de ~4.8% alrededor de 8 líneas) — consistente con la hipótesis de que muy poco historial crediticio es señal de riesgo, no de bajo riesgo. La zona media (9-25 líneas) se mantiene relativamente estable, sin una tendencia creciente clara. A partir de 29 líneas (114 casos y menos), el volumen se vuelve demasiado bajo para ser confiable, con oscilaciones erráticas (0%-50%). Se capará la variable en 29 y se creará un flag de anómalo para preservar la señal de "cliente con volumen de crédito atípicamente alto" sin dejar que el ruido estadístico distorsione el binning.

Variable: NumberOfTimes90DaysLate

In [63]:
df[~df['NumberOfTimes90DaysLate'].isin([96, 98])].groupby('NumberOfTimes90DaysLate')['SeriousDlqin2yrs'].agg(['mean', 'count'])

,mean,count
NumberOfTimes90DaysLate,,
0,0.046265,141661
1,0.336639,5243
2,0.499035,1555
3,0.577211,667
4,0.670103,291
5,0.633588,131
6,0.600000,80
7,0.815789,38
8,0.714286,21


In [64]:
df['NumberOfTimes90DaysLate_anomalo'] = (df['NumberOfTimes90DaysLate'] >= 5).astype(int)
df['NumberOfTimes90DaysLate_capped'] = df['NumberOfTimes90DaysLate'].clip(upper=5)

In [65]:
# 1. Flag específico para los códigos de sistema (esto ya deberías tenerlo de antes,
#    pero lo repito aquí para que quede todo junto y sea coherente)
df['NumberOfTimes90DaysLate_codigo_sistema'] = df['NumberOfTimes90DaysLate'].isin([96, 98]).astype(int)

# 2. Flag de "atraso alto" — SOLO para valores reales >= 5, excluyendo los códigos de sistema
df['NumberOfTimes90DaysLate_anomalo'] = (
    (df['NumberOfTimes90DaysLate'] >= 5) & (~df['NumberOfTimes90DaysLate'].isin([96, 98]))
).astype(int)

# 3. Versión capada — primero capamos a 5, pero solo tiene sentido para los valores reales.
#    Los códigos de sistema los reemplazamos por un valor "neutro" (ej. NaN o el propio cap),
#    ya que no queremos que "5" represente dos cosas distintas.
df['NumberOfTimes90DaysLate_capped'] = df['NumberOfTimes90DaysLate'].clip(upper=5)
df.loc[df['NumberOfTimes90DaysLate_codigo_sistema'] == 1, 'NumberOfTimes90DaysLate_capped'] = np.nan

In [66]:
df[['NumberOfTimes90DaysLate', 'NumberOfTimes90DaysLate_codigo_sistema', 
    'NumberOfTimes90DaysLate_anomalo', 'NumberOfTimes90DaysLate_capped']].drop_duplicates().sort_values('NumberOfTimes90DaysLate')

,NumberOfTimes90DaysLate,NumberOfTimes90DaysLate_codigo_sistema,NumberOfTimes90DaysLate_anomalo,NumberOfTimes90DaysLate_capped
0,0,0,0,0.0
2,1,0,0,1.0
186,2,0,0,2.0
13,3,0,0,3.0
1713,4,0,0,4.0
1298,5,0,1,5.0
3400,6,0,1,5.0
3929,7,0,1,5.0
5684,8,0,1,5.0
2910,9,0,1,5.0


> **Conclusión — `NumberOfTimes90DaysLate`:** es la variable con la relación más fuerte del EDA. Excluyendo los códigos de sistema (96/98), la mora sube de 4.6% (0 atrasos) a 33.7% (1 atraso) y sigue creciendo hasta estabilizarse alrededor del 60-80% a partir de 5+ atrasos, aunque con volumen cada vez más bajo. Se capa la variable en 5 (131 casos, último punto con volumen mínimamente estable) mediante `NumberOfTimes90DaysLate_capped`, con un flag `_anomalo` para atrasos reales de 5+. Los códigos de sistema (96/98) se tratan por separado con el flag `_codigo_sistema`, y su valor capado se marca como missing para evitar que se mezclen con los atrasos reales de 5+ en el binning de la Fase 2.